In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from rich.console import Console
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [68]:
todos = []
finished = []

In [4]:
def get_todos_report():
    if todos:
       
        result = ""
        for index, todo in enumerate(todos):
            print("the index is ", index)
            
            if finished[index]:
                
                result+= f"Todo number #{index+1}: [green][strike]{todo}[/strike][/green]\n" 
            else:
                result+= f"Todo number #{index+1}: [red]{todo}[/red]\n"

        show(result)
    else:
        return {}
    

In [5]:
def add_todos(todo_list: list[str]) -> str:
    
    todos.extend(todo_list)
    finished.extend(len(todo_list) * [False])
        
    return get_todos_report()

In [43]:
#add_todos(["buy groceries", "walk the dog", "drink planty of water"])

In [7]:
def mark_todos(todo_number):
    if 1<= todo_number <= len(todos):
        finished[todo_number-1] = True
       
        return get_todos_report()
        
    else:
        return "there is no existing todo on this number"

In [8]:
mark_todos(2)

the index is  0
the index is  1
the index is  2


Todo number #1: buy groceries
Todo number #2: walk the dog
Todo number #3: drink planty of water

In [37]:
create_todos_json = {
    "name" : "add_todos",
    "description": "new todos gets listed here and it returns a printed version of the todos naming the todo number, the name of the todo (what is to do) and marks it red if the todo hasn't been finished or green crossed if the todo was finished", 
    "parameters": {
        "type" : "object",
        "properties" : {
            "todo_list": {
                "type" : "array",
                "items": {"type": "string"},
                "title": "todo_list"
        }
        }
        ,
        "required": ["todo_list"],
        "additionalProperties": False
    }
}

#BadRequestError: Error code: 400 - {'error': {'message': "Invalid schema for function 'add_todos': 'todo_list' is not of type 'object', 'boolean'.", 'type': 'invalid_request_error', 'param': 'tools[1].function.parameters', 'code': 'invalid_function_parameters'}}


In [ ]:
create_mark_todos_json = {
    "name": "mark_todos",
    "description": "You can enter the number of the todo. If the todo with this number exist than it returns a printed version of the todos naming the todo number, the name of the todo (what is to do) and marks it red if the todo hasn't been finished or green crossed if the todo was finished",
    
    "parameters": {
        "type": "object",
        "properties": {
            "todo_number": {
                "description": "the 1 based index of the todo to mark as complete",
                "title": "todo_number",
                "type": "integer",

                
            }

        },
    "required": ["todo_number"],
    
    "additionalProperties": False
    }

}


In [38]:
tools = [{"type":"function", "function": create_mark_todos_json},
{"type": "function", "function": create_todos_json}]

In [12]:
def handle_tools(tool_calls):
    results = []
    for tool_call in tool_calls:
        print("I want to see what is in the tool_call", tool_call)
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results



In [39]:
openai = OpenAI()

In [87]:
import os
from dotenv import load_dotenv

google_api_key = os.getenv('GOOGLE_API_KEY')

gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"
def loop(messages):
    done = False

    while not done:
        response = openai.chat.completions.create(model = "gpt-5-mini", messages = messages, tools = tools)
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tools(tool_calls)
            messages.append(message)
            messages.extend(results)

        else:
            done = True
        
    show(response.choices[0].message.content)
    return response.choices[0].message.content
            


In [88]:
system_prompt = """ 
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
""" 
user_message= """Du möchtest italienisch lernen und du hast das Level B1 oder etwas darunter. Nun möchtest du gerne die Serie Ozark auf italienisch ansehen.
Nun möchtest du vorab gerne Wörter zur Verfügung gestellt bekommen, welche  in der dritten Folge dran kommen. Es sollen keine einfachen 
Wörter auf der Liste landen, sondern lediglich ab B1 oder höher. Suche auf opensubtitles.com nach:
Ozark S01E03 Italian subtitle WEB-DL Netflix
 nach:
Ozark Season 1 Episode 3 Italian subtitles
Suche passende italienische .srt-Untertitel die synchron sind. Wenn du passende Untertitel gefunden hast dann erstelle aus den Wörtern die B1 oder höher sind eine PDF Datei für mich"""

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_message}]

In [89]:
results = loop(messages)


I want to see what is in the tool_call ChatCompletionMessageFunctionToolCall(id='call_4gO5HKXpg32CHHjVBFeytmbv', function=Function(arguments='{"todo_list":["1) Plan erstellen: Schritte zum Finden, Prüfen und Verarbeiten der Untertitel definieren","2) Opensubtitles durchsuchen nach \'Ozark S01E03 Italian subtitle WEB-DL Netflix\' und \'Ozark Season 1 Episode 3 Italian subtitles\' und passende .srt herunterladen","3) Prüfen, ob die .srt synchron zur WEB‑DL/Netflix-Version ist; wenn nötig alternative Releases suchen","4) Aus der synchronen .srt alle Wörter extrahieren, die CEFR Niveau B1 oder höher sind (Lemmatisierung + Filtern gegen Wortlisten)","5) Aus der gefilterten Wortliste eine ansprechende PDF mit Übersetzungen, Beispielsätzen und Lerntipps erstellen","6) PDF bereitstellen / dem Nutzer anbieten, die Datei herunterzuladen oder Alternativen erklären"]}', name='add_todos'), type='function')
the index is  0
the index is  1
the index is  2
the index is  3
the index is  4
the index is 

Todo number #1: Plan erstellen: Anforderungen zusammenfassen (Episode 3, Italienisch, B1+ Vokabelliste, Deutsch 
erklärt, PDF-Layout).
Todo number #2: Datenbeschaffung abschätzen: Keine Web-/PDF-Tools verfügbar → realistische Alternative liefern 
(strukturierte Liste + druckfertiges Layout, Anleitung zum PDF-Export).
Todo number #3: Wortschatz für Ozark S01E03 (italienische Untertitel) thematisch ableiten: 
Kriminalität/Geldwäsche/Behörden/Familie; nur B1+ auswählen.
Todo number #4: Für jedes Wort: deutsche Erklärung (keine 1:1-Übersetzung, sondern kurze Definition/Erklärung), 
ggf. Beispielsatz kurz.
Todo number #5: Tabellarisches Layout erstellen: links IT, rechts DE-Erklärung, druckfertig (A4), Rich-Markup.
Todo number #6: Abschluss: Exporthinweis zu PDF (Drucken→Als PDF speichern).
Todo number #7: Finde die passenden italienischen Untertitel für Ozark S01E03 auf opensubtitles.com.
Todo number #8: Überprüfe die Synchronität der gefundenen Untertitel.
Todo number #9: Extrahiere Wörter ab B1-Niveau aus den Untertiteln.
Todo number #10: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #11: Finde passende italienische Untertitel für Ozark S01E03 auf opensubtitles.com, die synchron zur 
WEB-DL Netflix Version sind.
Todo number #12: Extrahiere Wörter aus den Untertiteln, die dem B1- oder höheren Sprachniveau entsprechen.
Todo number #13: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #14: 1) Plan erstellen: Schritte zum Finden, Prüfen und Verarbeiten der Untertitel definieren
Todo number #15: 2) Opensubtitles durchsuchen nach 'Ozark S01E03 Italian subtitle WEB-DL Netflix' und 'Ozark Season
1 Episode 3 Italian subtitles' und passende .srt herunterladen
Todo number #16: 3) Prüfen, ob die .srt synchron zur WEB‑DL/Netflix-Version ist; wenn nötig alternative Releases 
suchen
Todo number #17: 4) Aus der synchronen .srt alle Wörter extrahieren, die CEFR Niveau B1 oder höher sind 
(Lemmatisierung + Filtern gegen Wortlisten)
Todo number #18: 5) Aus der gefilterten Wortliste eine ansprechende PDF mit Übersetzungen, Beispielsätzen und 
Lerntipps erstellen
Todo number #19: 6) PDF bereitstellen / dem Nutzer anbieten, die Datei herunterzuladen oder Alternativen erklären

I want to see what is in the tool_call ChatCompletionMessageFunctionToolCall(id='call_YWK1RIEs0yio9zSCox6pR5h7', function=Function(arguments='{"todo_number":1}', name='mark_todos'), type='function')
the index is  0
the index is  1
the index is  2
the index is  3
the index is  4
the index is  5
the index is  6
the index is  7
the index is  8
the index is  9
the index is  10
the index is  11
the index is  12
the index is  13
the index is  14
the index is  15
the index is  16
the index is  17
the index is  18


Todo number #1: Plan erstellen: Anforderungen zusammenfassen (Episode 3, Italienisch, B1+ Vokabelliste, Deutsch 
erklärt, PDF-Layout).
Todo number #2: Datenbeschaffung abschätzen: Keine Web-/PDF-Tools verfügbar → realistische Alternative liefern 
(strukturierte Liste + druckfertiges Layout, Anleitung zum PDF-Export).
Todo number #3: Wortschatz für Ozark S01E03 (italienische Untertitel) thematisch ableiten: 
Kriminalität/Geldwäsche/Behörden/Familie; nur B1+ auswählen.
Todo number #4: Für jedes Wort: deutsche Erklärung (keine 1:1-Übersetzung, sondern kurze Definition/Erklärung), 
ggf. Beispielsatz kurz.
Todo number #5: Tabellarisches Layout erstellen: links IT, rechts DE-Erklärung, druckfertig (A4), Rich-Markup.
Todo number #6: Abschluss: Exporthinweis zu PDF (Drucken→Als PDF speichern).
Todo number #7: Finde die passenden italienischen Untertitel für Ozark S01E03 auf opensubtitles.com.
Todo number #8: Überprüfe die Synchronität der gefundenen Untertitel.
Todo number #9: Extrahiere Wörter ab B1-Niveau aus den Untertiteln.
Todo number #10: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #11: Finde passende italienische Untertitel für Ozark S01E03 auf opensubtitles.com, die synchron zur 
WEB-DL Netflix Version sind.
Todo number #12: Extrahiere Wörter aus den Untertiteln, die dem B1- oder höheren Sprachniveau entsprechen.
Todo number #13: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #14: 1) Plan erstellen: Schritte zum Finden, Prüfen und Verarbeiten der Untertitel definieren
Todo number #15: 2) Opensubtitles durchsuchen nach 'Ozark S01E03 Italian subtitle WEB-DL Netflix' und 'Ozark Season
1 Episode 3 Italian subtitles' und passende .srt herunterladen
Todo number #16: 3) Prüfen, ob die .srt synchron zur WEB‑DL/Netflix-Version ist; wenn nötig alternative Releases 
suchen
Todo number #17: 4) Aus der synchronen .srt alle Wörter extrahieren, die CEFR Niveau B1 oder höher sind 
(Lemmatisierung + Filtern gegen Wortlisten)
Todo number #18: 5) Aus der gefilterten Wortliste eine ansprechende PDF mit Übersetzungen, Beispielsätzen und 
Lerntipps erstellen
Todo number #19: 6) PDF bereitstellen / dem Nutzer anbieten, die Datei herunterzuladen oder Alternativen erklären

I want to see what is in the tool_call ChatCompletionMessageFunctionToolCall(id='call_U2dWHSZrUzcN5VNZ9ogyZYEH', function=Function(arguments='{"todo_number":2}', name='mark_todos'), type='function')
the index is  0
the index is  1
the index is  2
the index is  3
the index is  4
the index is  5
the index is  6
the index is  7
the index is  8
the index is  9
the index is  10
the index is  11
the index is  12
the index is  13
the index is  14
the index is  15
the index is  16
the index is  17
the index is  18


...

I want to see what is in the tool_call ChatCompletionMessageFunctionToolCall(id='call_c1NJd2z6wmnW70W3D2ZXOzr9', function=Function(arguments='{"todo_number":5}', name='mark_todos'), type='function')
the index is  0
the index is  1
the index is  2
the index is  3
the index is  4
the index is  5
the index is  6
the index is  7
the index is  8
the index is  9
the index is  10
the index is  11
the index is  12
the index is  13
the index is  14
the index is  15
the index is  16
the index is  17
the index is  18


Todo number #1: Plan erstellen: Anforderungen zusammenfassen (Episode 3, Italienisch, B1+ Vokabelliste, Deutsch 
erklärt, PDF-Layout).
Todo number #2: Datenbeschaffung abschätzen: Keine Web-/PDF-Tools verfügbar → realistische Alternative liefern 
(strukturierte Liste + druckfertiges Layout, Anleitung zum PDF-Export).
Todo number #3: Wortschatz für Ozark S01E03 (italienische Untertitel) thematisch ableiten: 
Kriminalität/Geldwäsche/Behörden/Familie; nur B1+ auswählen.
Todo number #4: Für jedes Wort: deutsche Erklärung (keine 1:1-Übersetzung, sondern kurze Definition/Erklärung), 
ggf. Beispielsatz kurz.
Todo number #5: Tabellarisches Layout erstellen: links IT, rechts DE-Erklärung, druckfertig (A4), Rich-Markup.
Todo number #6: Abschluss: Exporthinweis zu PDF (Drucken→Als PDF speichern).
Todo number #7: Finde die passenden italienischen Untertitel für Ozark S01E03 auf opensubtitles.com.
Todo number #8: Überprüfe die Synchronität der gefundenen Untertitel.
Todo number #9: Extrahiere Wörter ab B1-Niveau aus den Untertiteln.
Todo number #10: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #11: Finde passende italienische Untertitel für Ozark S01E03 auf opensubtitles.com, die synchron zur 
WEB-DL Netflix Version sind.
Todo number #12: Extrahiere Wörter aus den Untertiteln, die dem B1- oder höheren Sprachniveau entsprechen.
Todo number #13: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #14: 1) Plan erstellen: Schritte zum Finden, Prüfen und Verarbeiten der Untertitel definieren
Todo number #15: 2) Opensubtitles durchsuchen nach 'Ozark S01E03 Italian subtitle WEB-DL Netflix' und 'Ozark Season
1 Episode 3 Italian subtitles' und passende .srt herunterladen
Todo number #16: 3) Prüfen, ob die .srt synchron zur WEB‑DL/Netflix-Version ist; wenn nötig alternative Releases 
suchen
Todo number #17: 4) Aus der synchronen .srt alle Wörter extrahieren, die CEFR Niveau B1 oder höher sind 
(Lemmatisierung + Filtern gegen Wortlisten)
Todo number #18: 5) Aus der gefilterten Wortliste eine ansprechende PDF mit Übersetzungen, Beispielsätzen und 
Lerntipps erstellen
Todo number #19: 6) PDF bereitstellen / dem Nutzer anbieten, die Datei herunterzuladen oder Alternativen erklären

I want to see what is in the tool_call ChatCompletionMessageFunctionToolCall(id='call_1XGslrcx6uZx2BBOWw7t4oaX', function=Function(arguments='{"todo_number":6}', name='mark_todos'), type='function')
the index is  0
the index is  1
the index is  2
the index is  3
the index is  4
the index is  5
the index is  6
the index is  7
the index is  8
the index is  9
the index is  10
the index is  11
the index is  12
the index is  13
the index is  14
the index is  15
the index is  16
the index is  17
the index is  18


Todo number #1: Plan erstellen: Anforderungen zusammenfassen (Episode 3, Italienisch, B1+ Vokabelliste, Deutsch 
erklärt, PDF-Layout).
Todo number #2: Datenbeschaffung abschätzen: Keine Web-/PDF-Tools verfügbar → realistische Alternative liefern 
(strukturierte Liste + druckfertiges Layout, Anleitung zum PDF-Export).
Todo number #3: Wortschatz für Ozark S01E03 (italienische Untertitel) thematisch ableiten: 
Kriminalität/Geldwäsche/Behörden/Familie; nur B1+ auswählen.
Todo number #4: Für jedes Wort: deutsche Erklärung (keine 1:1-Übersetzung, sondern kurze Definition/Erklärung), 
ggf. Beispielsatz kurz.
Todo number #5: Tabellarisches Layout erstellen: links IT, rechts DE-Erklärung, druckfertig (A4), Rich-Markup.
Todo number #6: Abschluss: Exporthinweis zu PDF (Drucken→Als PDF speichern).
Todo number #7: Finde die passenden italienischen Untertitel für Ozark S01E03 auf opensubtitles.com.
Todo number #8: Überprüfe die Synchronität der gefundenen Untertitel.
Todo number #9: Extrahiere Wörter ab B1-Niveau aus den Untertiteln.
Todo number #10: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #11: Finde passende italienische Untertitel für Ozark S01E03 auf opensubtitles.com, die synchron zur 
WEB-DL Netflix Version sind.
Todo number #12: Extrahiere Wörter aus den Untertiteln, die dem B1- oder höheren Sprachniveau entsprechen.
Todo number #13: Erstelle eine PDF-Datei mit den extrahierten Wörtern.
Todo number #14: 1) Plan erstellen: Schritte zum Finden, Prüfen und Verarbeiten der Untertitel definieren
Todo number #15: 2) Opensubtitles durchsuchen nach 'Ozark S01E03 Italian subtitle WEB-DL Netflix' und 'Ozark Season
1 Episode 3 Italian subtitles' und passende .srt herunterladen
Todo number #16: 3) Prüfen, ob die .srt synchron zur WEB‑DL/Netflix-Version ist; wenn nötig alternative Releases 
suchen
Todo number #17: 4) Aus der synchronen .srt alle Wörter extrahieren, die CEFR Niveau B1 oder höher sind 
(Lemmatisierung + Filtern gegen Wortlisten)
Todo number #18: 5) Aus der gefilterten Wortliste eine ansprechende PDF mit Übersetzungen, Beispielsätzen und 
Lerntipps erstellen
Todo number #19: 6) PDF bereitstellen / dem Nutzer anbieten, die Datei herunterzuladen oder Alternativen erklären

Plan (To‑do) — Status
- 1) Plan erstellen: erledigt. ✔️
- 2) opensubtitles.com nach "Ozark S01E03 Italian subtitle WEB-DL Netflix" und "Ozark Season 1 Episode 3 Italian 
subtitles" durchsuchen und passende .srt herunterladen: nicht ausgeführt — ich habe hier keinen direkten 
Webzugriff, daher konnte ich die Seite nicht durchsuchen. ❌
- 3) Prüfen, ob die .srt synchron ist; bei Bedarf alternative Releases suchen: nicht ausgeführt (kein Zugriff). ❌
- 4) Aus synchroner .srt alle Wörter extrahieren, die CEFR B1 oder höher sind: kann ich automatisiert machen, 
sobald du mir die .srt-Datei zur Verfügung stellst. Bis dahin erstelle ich eine geschätzte Übungsliste. ❌ (wartet 
auf .srt)
- 5) Aus der gefilterten Wortliste eine PDF mit Übersetzungen, Beispielsätzen und Lerntipps erstellen: kann ich 
erstellen, sobald die echte Wortliste vorliegt; unten findest du bereits ein vollständiges PDF‑fertiges 
Inhaltslayout (als Text), das du sofort in ein PDF speichern kannst. ❌ (erst nach .srt möglich für exakte Liste)
- 6) PDF bereitstellen / Download anbieten: möglich, wenn du die .srt hochlädst; alternativ gebe ich dir sofort das
vorbereitete PDF‑Inhaltspaket zur direkten Verwendung. ❌ (erst nach .srt möglich für exakte Version)

Wichtig zu meiner Einschränkung
- Ich kann hier keine Webseiten durchsuchen oder Dateien von opensubtitles.com herunterladen. Wenn du möchtest, 
lade bitte die passende .srt-Datei (Ozark S01E03 Italian, WEB-DL Netflix oder eine andere WEB‑DL/Netflix‑Release) 
hier hoch — ich verarbeite sie dann automatisiert, filtere nur B1+ Wörter und erstelle die PDF für dich.
- Falls du die Datei selbst herunterladen willst: die genauen Suchbegriffe, die du verwenden kannst:
  - "Ozark S01E03 Italian subtitle WEB-DL Netflix"
  - "Ozark Season 1 Episode 3 Italian subtitles"
  - Suche nach Releases mit "WEB-DL", "Netflix", "WEBRip" — ideal ist eine Datei, die ausdrücklich mit "Netflix" 
oder "WEB-DL" beschrieben ist, damit sie synchron zu deiner Version ist.

Kurzanleitung: Wie du eine Synchronität der Untertitel prüfst
1. Öffne die Episode in deinem Player (z. B. VLC) und lade die .srt dazu.
2. Prüfe die erste Zeile der .srt: beginnt sie kurz nach dem Szenenanfang (z. B. <00:00:05,000>)? Wenn die Zeiten 
deutlich abweichen (z. B. erste Untertitel bei 00:02:00), ist die Datei vermutlich für eine andere Release-Version.
3. Springe in einer markanten Szene (ein Dialog, der in der Episode klar zu erkennen ist) und vergleiche, ob Text 
und Ton/Sprachfluss passen. Kleinere Verschiebungen lassen sich mit VLC → Untertitel → Synchronisierung 
korrigieren; größere Abweichungen sind problematisch.
4. Wenn du unsicher bist: nimm eine kurze Videosequenz (30–60 s), notiere Zeitbereich, und wir können die .srt 
prüfen, sobald du sie hochlädst.

Wenn du die .srt hochlädst: Was ich automatisch tun werde
- Tokenisierung + Lemmatisierung (italienisch).
- Entfernen von Füllwörtern, Interjektionen, Eigennamen (falls du das möchtest).
- Abgleich mit einer CEFR‑Wortliste für Italienisch (A1–C2). Ich filtere nur Wörter ab B1 aufwärts.
- Sortieren nach Häufigkeit und Relevanz (Kontexthäufigkeit im Episoden‑Transcript).
- Für jede Zielvokabel: Grundform, Wortart, deutsche Übersetzung, ein kurzes Beispielsatz aus dem Untertitel 
(kontextnah) oder ein eigener Beispielsatz, Hinweise zur Wortverwendung.
- Ausgabe als druckfertiges PDF (Titelblatt, Inhaltsverzeichnis, Wortliste, Lernhinweise, evtl. thematische 
Cluster).

Praktische Anleitung, falls du es selbst machen willst
- Wenn du die .srt hast, kannst du das Word/PDF‑Doku einfach manuell erstellen:
  1. Untertitel im Texteditor öffnen, alle Zeitstempel entfernen (z. B. mithilfe einer einfachen Regex).
  2. Die reine Textdatei in eine Sprachbearbeitung (z. B. mit Python + spaCy-it) lemmatisieren und Token 
extrahieren.
  3. Gegen eine CEFR‑Liste filtern (z. B. öffentlich verfügbare Listen benutzen).
  4. Ergebnis in Google Docs oder LibreOffice einfügen → Datei → Als PDF exportieren.

Beispiel-PDF‑Inhalt (

In [83]:
results


'Ich werde nun die passenden italienischen Untertitel für Ozark S01E03 finden und die Wörter extrahieren, die dem B1- oder höheren Sprachniveau entsprechen. Anschließend werde ich eine PDF-Datei mit diesen Wörtern erstellen.\n'

In [62]:
system_prom = """Erstelle aus folgendem Vokabeltext eine PDF-Datei.

Verwende Python und reportlab.
Format:
- Titel auf der ersten Seite
- Zweispaltige Tabelle
- Linke Spalte: Italienisch
- Rechte Spalte: Deutsche Erklärung
- Automatischer Zeilenumbruch
- A4 Hochformat

Extrahiere die Vokabeln.

Gib ausschließlich JSON zurück.

Format:

{
  "title": "...",
  "vocabulary": [
    {
      "italian": "...",
      "german": "..."
    }
  ]
}

Hier ist der Inhalt der LLM bitte filtere nur die italienischen Wörter + Erklärung heraus und streiche den Rest:"""+ results

In [64]:
import os
from dotenv import load_dotenv

google_api_key = os.getenv('GOOGLE_API_KEY')

gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

messages = [{"role": "user", "content": system_prom}]

res = gemini.chat.completions.create(model = model_name, messages = messages)

ans = res.choices[0].message.content



In [65]:
print(res
)

ChatCompletion(id='No8haqMK5p-R1Q-R6O_oCw', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='```json\n{\n  "title": "Wortschatz (B1+) – Ozark auf Italienisch (voraussichtlich Episode 3 thematisch passend)",\n  "vocabulary": [\n    {\n      "italian": "riciclaggio di denaro",\n      "german": "Geldwäsche; illegal „gereinigtes“ Geld in den legalen Kreislauf bringen"\n    },\n    {\n      "italian": "cartello",\n      "german": "Drogenkartell; kriminelle Organisation (v. a. Drogenhandel)"\n    },\n    {\n      "italian": "capo / boss",\n      "german": "Anführer einer kriminellen Gruppe; „der Chef“"\n    },\n    {\n      "italian": "sicario",\n      "german": "Auftragsmörder"\n    },\n    {\n      "italian": "estorsione",\n      "german": "Erpressung; Geld/Handlungen durch Drohung erzwingen"\n    },\n    {\n      "italian": "minaccia",\n      "german": "Drohung; Ankündigung von Gewalt/Schaden"\n    },\n    {\n      "italian": "ricatto",\